# TML — A100 RapidOCR Generic Watcher
Reusable for any year. Tuned A100 profile: 12 CUDA workers, 8 VPS/SFTP downloaders, zero Gallica requests. Claim rows carry their own remote_cache so VPS can switch stages without restarting Colab.


In [ ]:
YEAR=1904
WORKERS=12
DOWNLOADERS=8
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
CLAIM=f'{BASE}/00_MANIFEST/colab_active_claims.tsv'
STOP=f'{BASE}/00_MANIFEST/colab_rapid_stop_{YEAR}.flag'
print('CONFIG',YEAR,'RapidOCR workers',WORKERS,'downloaders',DOWNLOADERS,'claim',CLAIM,flush=True)


In [ ]:
import os,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
import onnxruntime as ort
providers=ort.get_available_providers(); print('ORT',ort.__version__,'providers',providers,flush=True)
if 'CUDAExecutionProvider' not in providers: raise RuntimeError('CUDAExecutionProvider unavailable')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
print('RAPID_ENV_READY',flush=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'; open(KEY_FILE,'wb').write(data); os.chmod(KEY_FILE,0o600)
print('KEY_FILE_READY',name,flush=True)


In [ ]:
import subprocess,sys
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip(); print('CODE',commit,flush=True)
cmd=[sys.executable,'-u',f'{REPO}/colab/rapid_watch_filekey.py','--vps-key-file',KEY_FILE,'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--claim',CLAIM,'--stop-flag',STOP,'--workers',str(WORKERS),'--downloaders',str(DOWNLOADERS),'--poll','10']
print('STARTING_RAPID_WATCH year',YEAR,'workers',WORKERS,'reference_ppm=72.948',flush=True)
subprocess.run(cmd,check=True)
